# Baltic SST — Yearly Comparison

This script is part of the shared group project for the course **Open Science in Practice for Coastal Ocean Data Analysis and Visualization** Co-organized by [MAGICA](https://magica-project.eu/), [CMCC](https://www.cmcc.it/), the [University of Bologna](https://www.unibo.it/en), and [EGI](https://www.egi.eu/).

* It reads Baltic SST data from Copernicus, computes monthly means per year,
and provides interactive plots for inspection:
    1. **Spatial map** — monthly mean SST with a time slider
    2. **Seasonal cycle** — area-mean SST for each year

`version 2.1.0` `23.04.2026`

---

In [ ]:
import os
import warnings
import operator
from functools import reduce

import xarray as xr
import pandas as pd
import copernicusmarine
import hvplot.xarray
import hvplot.pandas

import panel as pn
pn.extension('bokeh')

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

## Fetch from Copernicus Marine & cache to Zarr

Skipped automatically if zarr file in the `CACHE_PATH` already exists.
If not, uses a Coiled cluster for fast parallel computation.

In [ ]:
from dotenv import load_dotenv
import fsspec

load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env', override=True)

fs = fsspec.filesystem(
    's3',
    anon=False,
    skip_instance_cache=True,
    key=os.environ['AWS_ACCESS_KEY_ID'],
    secret=os.environ['AWS_SECRET_ACCESS_KEY'],
    endpoint_url=os.environ['ENDPOINT_URL'],
)

username = 'moeindst'
CACHE_S3 = f's3://{username}/baltic_sst_monthly.zarr'

In [ ]:
CACHE_S3

In [ ]:
os.environ['ENDPOINT_URL']

In [ ]:
%%time
try:
    cache_exists = len(fs.ls(CACHE_S3)) > 0
except (FileNotFoundError, PermissionError):
    cache_exists = False

if not cache_exists:
    import coiled

    dataset_id = 'DMI-BALTIC-SST-L4-NRT-OBS_FULL_TIME_SERIE'
    ds = copernicusmarine.open_dataset(dataset_id)
    ds_baltic = ds.sel(latitude=slice(54, 66), longitude=slice(8, 30))

    region = 'eu-central-1'
    cluster = coiled.Cluster(
        region=region,
        arm=True,
        worker_vm_types=['t4g.large'],
        n_workers=30,
        name=f'moeindst_{region}',
        wait_for_workers=True,
        compute_purchase_option='spot_with_fallback',
        software='protocoast-notebook-arm',
        workspace='esip-lab',
        timeout=360,
    )
    client = cluster.get_client()

    da_monthly_raw = (
        ds_baltic['analysed_sst']
        .resample(time='ME')
        .mean()
        .load()
    ) - 273.15
    da_monthly_raw.attrs['units'] = 'degC'

    # clear stale encoding from Copernicus source — scale_factor/add_offset/
    # _FillValue get re-applied on reload and turn all values to NaN
    da_monthly_raw.encoding.clear()
    ds_to_save = da_monthly_raw.to_dataset()
    for v in list(ds_to_save.coords):
        ds_to_save[v].encoding.clear()

    ds_to_save.to_zarr(fs.get_mapper(CACHE_S3), mode='w')
    print(f'Saved to {CACHE_S3}')
    client.close()
    cluster.close()
else:
    print(f'Cache found at {CACHE_S3} — skipping fetch.')

In [ ]:
ds_baltic

## Load from Zarr

In [ ]:
%%time
da_monthly = xr.open_zarr(fs.get_mapper(CACHE_S3))['analysed_sst'].load()
# da_monthly

## Split by year & build seasonal DataFrame

In [ ]:
%%time
years = sorted({int(t) for t in da_monthly.time.dt.year.values})
da_by_year = {yr: da_monthly.sel(time=str(yr)) for yr in years}

records = {}
for yr, da in da_by_year.items():
    area_mean = da.mean(dim=['latitude', 'longitude']).values
    months = da.time.dt.month.values
    records[str(yr)] = pd.Series(area_mean, index=months)

df_seasonal = pd.DataFrame(records)
df_seasonal.index.name = 'month'

In [ ]:
df_seasonal

## Plot 1 — Spatial monthly SST map with time slider

In [ ]:
import cmocean

In [ ]:
%%time
map_opts = dict(
    x='longitude', y='latitude',
    rasterize=True, geo=True,
    cmap=cmocean.cm.thermal, clim=(0, 20),
    tiles='OSM',
    width=700, height=500, alpha=0.95
)

plot1 = da_monthly.hvplot(
    title='Baltic SST Monthly Mean (degC)',
    groupby='time',
    **map_opts,
)
plot1

## Plot 2 — Seasonal cycle by year

In [ ]:
# COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
COLORS = ['royalblue', 'seagreen', 'orange', 'red']

curves = [
    df_seasonal[col].hvplot.line(
        label=col,
        color=COLORS[i % len(COLORS)],
        line_width=2,
    )
    for i, col in enumerate(df_seasonal.columns)
]

seasonal_plot = reduce(operator.mul, curves).opts(
    title='Baltic SST — Monthly spatial mean',
    xlabel='Month', ylabel='SST (degC)',
    legend_position='top_left',
    width=700, height=400,
    show_grid=True,
)
seasonal_plot

## Panel Dashboard

Serve interactively with:
```bash
panel serve baltic_sst_yearly_comparison_v2.ipynb --show
```
Opens at `http://localhost:5006`

In [ ]:
pn.Column(                                                                                                 
      pn.pane.Markdown('## Spatial Monthly Mean SST'),                                                       
      pn.panel(plot1),                                                                                       
      pn.layout.Divider(),                                                                                   
      pn.pane.Markdown('## Seasonal Cycle — Area-Mean SST by Year'),                                         
      pn.panel(seasonal_plot),                                                                               
  )

                                        ----------- End of Document -----------